In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Memory experiment analysis: min-distance + ROC curves.
Refactored into a single organized script. 
Modify only `run_experiment_at_noise` to explore new dynamics.
"""
%load_ext autoreload
%autoreload 2
# ===================== Imports =====================
import sys, os, glob, json, math, datetime
from collections import defaultdict

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression

# project-specific paths
sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

import DistanceMemoryModel
import encoders
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used
from utls.runners import run_experiment_scores, run_experiment_scores_itemwise, run_experiment_itemwise_hits_fas
from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime, find_optimal_roc_threshold
from utls.plotting import plot_across_noise, plot_noise_overlays, plot_histograms_all_models, plot_model_grid_summary
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, load_all_runs, save_runs_summary

import os, json
import numpy as np
from scipy.stats import norm
from sklearn.utils import resample
from utls.roc_utils import roc_from_arrays  # if separate, or inline

import math


import numpy as np
import torch
import pandas as pd
from collections import defaultdict

In [ ]:
old = False

save_base="/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/model-behavior_test/"

which_task = "atexts-len120"

seqs_paths = {
    "ind-nature-len120": "mem_exp_ind-nature_2025", 
    "global-music-len120": "global-music-2025-n_80",
    "atexts-len120": "mem_exp_atexts_2025",
    "nhs-region-len120": "nhs-region-n_80"
}

if old:
    t_results = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")
else:
    t_results = glob.glob(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}/*csv")

PC_DIMS = 256
DEVICE = 'cuda'
NV_DEFAULT = 0.1

# Encode clean reps
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=PC_DIMS, model_params=model_params,
    sr=20000, rms_level=0.05, duration=2.0, device=DEVICE
)

In [ ]:

if old:
    # OLD MULTI-ISI DATA
    files = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")
    print("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")
    exps, seqs, fnames_= load_results_with_exclusion_2(
        "/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/V15",
        min_dprime=2, min_trials=256, skip_len60=True, verbose=False, return_skipped=False
    )
    
    base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/sequences/"
    
    # Build experiment_list
    experiment_list = []
    for seq_ in seqs:
        with open(base_path + seq_, 'r') as f:
            data = json.load(f)
        stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/" + s 
                      for s in data['filenames_order']]
    
        experiment_list.append(stim_paths)
    
    all_files = sorted({fn for seq in experiment_list for fn in seq})
    name_to_idx = {fn: i for i, fn in enumerate(all_files)}
    
    tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
    tmp._fill_memory_bank(all_files)
    with torch.no_grad():
        X0 = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)

else:
    # SINGLE ISI DATA
    which_task = "atexts-len120"
    
    seqs_paths = {
        "ind-nature-len120": "mem_exp_ind-nature_2025", 
        "global-music-len120": "global-music-2025-n_80",
        "atexts-len120": "mem_exp_atexts_2025",
        "nhs-region-len120": "nhs-region-n_80"
    }
    base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/isi_16/len120/"
    exps, seqs, fnames = load_results_with_exclusion_2(
        f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}",
        min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
    )

    print(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_16/{which_task}")
    move_sequences_to_used(base_path.format(seqs_paths[which_task]), seqs)
    
    # Build experiment_list
    experiment_list = []
    for seq_ in seqs:
        with open(base_path.format(seqs_paths[which_task]) + seq_, 'r') as f:
            data = json.load(f)
        stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/" + seqs_paths[which_task] + "/" + s 
                      for s in data['filenames_order']]
        experiment_list.append(stim_paths)
    
    all_files = sorted({fn for seq in experiment_list for fn in seq})
    name_to_idx = {fn: i for i, fn in enumerate(all_files)}
    
    tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
    tmp._fill_memory_bank(all_files)
    with torch.no_grad():
        X0 = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)

In [ ]:
# # Build experiment_list
# experiment_list = glob.glob("/om2/data/public/audioset/wavs/eval_segments_downloads/eval_segments_downloads_0/*wav")[:5000]


# eval_reps = []
# for i in range(len(experiment_list)):
#     try:
#         eval_reps.append(pc_texture_model(experiment_list[i]))
#     except AttributeError:
#         continue

# final_eval_reps = []
# for eval_rep in eval_reps:
#     try:
#         eval_rep.detach().clone().view(-1)
#     except AttributeError:
#         continue
#     final_eval_reps.append(eval_rep)
        
# with torch.no_grad():
#     X_all = torch.stack([rep.detach().clone().view(-1) for rep in final_eval_reps], dim=0).to(DEVICE)

import os
import pandas as pd
import soundfile as sf  # or librosa
from tqdm import tqdm

csv_path = "/orcd/data/jhm/001/om2/bjmedina/chexture_choolbox/assets/OVERLAPPED_0.1s_balanced_sound_list_with_stationarity_score_no_speech_no_music_audioset_matlab_coch_rms0p02.csv"
base_dir = "/om2/data/public/audioset/wavs/balanced_train_segments_downloads/"
out_dir  = "/orcd/data/jhm/001/om2/bjmedina/tmp_stationary_segments/"  # temp folder for extracted clips

# os.makedirs(out_dir, exist_ok=True)

# df = pd.read_csv(csv_path)
# df_filt = df[df["stationarity_score"] <= -0.6].copy()
# print(f"{len(df_filt)} rows after filtering")


# SEG_DUR = 2.0  # seconds

# for _, row in tqdm(df_filt.iterrows(), total=len(df_filt)):
#     rel_path = row["filepath"]
#     onset = float(row["onset_time"])
#     src_path = os.path.join(base_dir, rel_path)
#     if not os.path.exists(src_path):
#         continue

#     try:
#         y, sr = sf.read(src_path)
#         start_sample = int(onset * sr)
#         end_sample = int((onset + SEG_DUR) * sr)
#         segment = y[start_sample:end_sample]

#         # Skip too-short clips
#         if len(segment) < sr:  # <1s of data
#             continue

#         # Create an informative name
#         fname = f"{os.path.splitext(os.path.basename(rel_path))[0]}_t{onset:.2f}.wav"
#         out_path = os.path.join(out_dir, fname)

#         sf.write(out_path, segment, sr)
#     except Exception as e:
#         print(f"Skipping {src_path}@{onset:.2f}: {e}")

wav_list = sorted([os.path.join(out_dir, f) for f in os.listdir(out_dir) if f.endswith(".wav")])
eval_reps = []

for path in tqdm(wav_list):
    try:
        rep = pc_texture_model(path)
        eval_reps.append(rep)
    except Exception as e:
        print(f"Skipping {path}: {e}")

with torch.no_grad():
    X_all = torch.stack([r.detach().clone().view(-1) for r in eval_reps], dim=0).to(DEVICE)

In [ ]:
human_runs = []
#print(t_results)
for result in t_results:
    df = pd.read_csv(result)
    main_exp = df[df['stim_type'] == 'main']
    if len(main_exp) < 256:
        continue
    human_runs.append(convert_human_to_model_struct(main_exp))

# Compute average human d′ vs ISI

if old:    
    isis_human = [0, 1, 2, 3, 4, 8, 16, 32, 64]
else:
    isis_human = [1, 17]
dprimes_human = []
for k in isis_human:
    aucs = []
    for run in human_runs:
        res = roc_for_isi(run, k)
        if res is not None:
            _, _, auc = res
            aucs.append(auroc_to_dprime(auc))
    dprimes_human.append(np.nanmean(aucs))
    
human_curve = np.array(dprimes_human, dtype=float)
human_curve = human_curve[~np.isnan(human_curve)]
print(dprimes_human)

In [ ]:
from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir

from utls.reliability import compute_power_curve
from utls.plotting import plot_power_curve

results_humans = compute_itemwise_split_half_reliability(exps, min_isi=min(isis_human), max_isi=max(isis_human))
hits = results_humans['itemwise_responses']['hits']
false_alarms  = results_humans['itemwise_responses']['false_alarms']

In [ ]:
from sklearn.neighbors import NearestNeighbors


fa_mean = false_alarms.mean(axis=0)   # mean FA rate per sound
fa_mean.name = "fa_mean"

X = X0.cpu().numpy() if hasattr(X0, "device") else X0
# Compute column-wise standard deviation
stds = X.std(axis=0, ddof=1)  # ddof=1 gives unbiased estimate
stds[stds == 0] = 1.0         # avoid divide-by-zero

# Scale each feature (z-score by std only)
X = X / stds


X_all = X_all.cpu().numpy() if hasattr(X_all, "device") else X_all
# Compute column-wise standard deviation
stds =X_all.std(axis=0, ddof=1)  # ddof=1 gives unbiased estimate
stds[stds == 0] = 1.0         # avoid divide-by-zero

# Scale each feature (z-score by std only)
X_all = X_all/ stds


k = 10 # usually use a bit larger k for prior density
knn_prior = NearestNeighbors(n_neighbors=k, metric='cosine').fit(X_all)
dists_prior, _ = knn_prior.kneighbors(X)

# lower mean distance = denser region (more typical)
prior_mean_dist = dists_prior.mean(axis=1)
prior_density = prior_mean_dist # convenient for regression

item_names = list(name_to_idx.keys())
print(prior_density.shape, len(item_names))


prior_df = pd.DataFrame({
    "item": item_names,
    "prior_density": prior_density,
    "fa_mean": [fa_mean.get(n.split('/')[-1], np.nan) for n in item_names]
}).dropna(subset=["fa_mean"])
prior_df = prior_df.dropna(subset=["fa_mean"])  # keep items with data

from scipy.stats import spearmanr, pearsonr

r_spearman, _ = spearmanr(prior_df["prior_density"], prior_df["fa_mean"])
r_pearson, _  = pearsonr(prior_df["prior_density"], prior_df["fa_mean"])

print(f"Spearman r = {r_spearman:.3f}, Pearson r = {r_pearson:.3f}")

import matplotlib.pyplot as plt

plt.figure(figsize=(6,5))
plt.scatter(prior_df["prior_density"], prior_df["fa_mean"], alpha=0.7, color='r')
plt.xlabel("Prior (mean kNN distance)")
plt.ylabel("False Alarm Rate")
plt.ylim([0,1])
plt.title(f"Prior vs False Alarms (Spearman r={r_spearman:.2f})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.stats import norm, spearmanr, pearsonr
import matplotlib.pyplot as plt

def compute_itemwise_dprime(hits, fas):
    """Compute d' per item with standard edge corrections."""
    h = hits.clip(1e-3, 1 - 1e-3)
    f = fas.clip(1e-3, 1 - 1e-3)
    return norm.ppf(h) - norm.ppf(f)

# --- 1. Mean hit and FA rate per sound ---
hit_mean = hits.mean(axis=0)
fa_mean  = false_alarms.mean(axis=0)

# --- 2. Compute d' per sound ---
dprime = compute_itemwise_dprime(hit_mean, fa_mean)


# --- 4. Combine into one DataFrame ---
item_names = list(name_to_idx.keys())
df = pd.DataFrame({
    "item": item_names,
    "prior_density": prior_density,
    "hit": [hit_mean.get(n.split("/")[-1], np.nan) for n in item_names],
    "fa":  [fa_mean.get(n.split("/")[-1],  np.nan) for n in item_names],
    "dprime": [dprime.get(n.split("/")[-1], np.nan) if isinstance(dprime, pd.Series)
               else np.nan for n in item_names]
})

# if dprime isn't a Series, vectorize instead
if not isinstance(dprime, pd.Series):
    df["dprime"] = compute_itemwise_dprime(df["hit"], df["fa"])

df = df.dropna(subset=["dprime"])

# --- 5. Correlation ---
r_spear, _ = spearmanr(df["prior_density"], df["dprime"])
r_pear, _  = pearsonr(df["prior_density"], df["dprime"])

print(f"Spearman r = {r_spear:.3f}, Pearson r = {r_pear:.3f}")

# --- 6. Plot ---
plt.figure(figsize=(6,5))
plt.scatter(df["prior_density"], df["dprime"], alpha=0.7, color='purple')
plt.xlabel(f"Prior (mean kNN distance [K={k}])")
plt.ylabel("Itemwise d′")
plt.title(f"Prior vs d′ (Pearson r={r_pear:.2f}) New Experiment")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
files = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")
exps_, seqs_, fnames_ = load_results_with_exclusion_2(
    "/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/V15",
    min_dprime=2, min_trials=120, skip_len60=True, verbose=False, return_skipped=False
)

base_path_ = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/sequences/"

# Build experiment_list
experiment_list_ = []
for seq_ in seqs_:
    with open(base_path_ + seq_, 'r') as f:
        data = json.load(f)
    stim_paths = ["/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_v15/" + s 
                  for s in data['filenames_order']]

    experiment_list_.append(stim_paths)

save_base="/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/model-behavior_v9/"
t_results = glob.glob("/mindhive/mcdermott/www/bjmedina/experiments/mem_exp_v02/results/*V15.csv")

PC_DIMS = 256
DEVICE = 'cuda'
NV_DEFAULT = 0.1

# Encode clean reps
pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=PC_DIMS, model_params=model_params,
    sr=20000, rms_level=0.05, duration=2.0, device=DEVICE
)


all_files_ = sorted({fn for seq in experiment_list_ for fn in seq})
name_to_idx_ = {fn: i for i, fn in enumerate(all_files_)}

tmp = DistanceMemoryModel.DistanceMemoryModel(pc_texture_model, NV_DEFAULT, criterion=0.0, device=DEVICE)
tmp._fill_memory_bank(all_files_)
with torch.no_grad():
    X0_ = torch.stack([rep.detach().clone().view(-1) for rep in tmp.memory_bank], dim=0).to(DEVICE)

In [ ]:
results_humans_early = compute_itemwise_split_half_reliability(exps_, min_isi=0, max_isi=8)
hits_early = results_humans_early['itemwise_responses']['hits']
#false_alarms_early  = results_humans_early['itemwise_responses']['false_alarms']

results_humans_late = compute_itemwise_split_half_reliability(exps_, min_isi=16, max_isi=64)
hits_late = results_humans_late['itemwise_responses']['hits']
false_alarms  = results_humans_late['itemwise_responses']['false_alarms']

# we have YT_ids but we dont want that...

In [ ]:
fa_mean  = false_alarms.mean(axis=0)
hit_early_mean = hits_early.mean(axis=0)
hit_late_mean  = hits_late.mean(axis=0)

X = X0_.cpu().numpy() if hasattr(X0_, "device") else X0_
item_names = list(name_to_idx_.keys())

k = 5
knn = NearestNeighbors(n_neighbors=k+1, metric='euclidean').fit(X)
dists, _ = knn.kneighbors(X)
mean_dist = dists[:, 1:].mean(axis=1)
typicality = -mean_dist  # higher = more typical

typ_df = pd.DataFrame({"item": item_names, "typicality": typicality})

In [ ]:
valid_items = set(hits_early.columns) | set(hits_late.columns) \
             | set(false_alarms.columns)
print(len(valid_items))

item_names = np.array(list(name_to_idx_.keys()))
typicality = np.array(typicality)

# keep only items that exist in valid_items
keep_mask = np.array([n.split('/')[-1] in valid_items for n in item_names])
item_names = item_names[keep_mask]
typicality = typicality[keep_mask]

In [ ]:
# compute itemwise means (as you already do)
fa_mean  = false_alarms.mean(axis=0)
hit_early_mean = hits_early.mean(axis=0)
hit_late_mean  = hits_late.mean(axis=0)

# merge only where all exist
df = pd.DataFrame({
    "item": item_names,
    "typicality": typicality,
    "fa":  [fa_mean.get(n.split("/")[-1], np.nan) for n in item_names],
    "hit_early": [hit_early_mean.get(n.split("/")[-1], np.nan) for n in item_names],
    "hit_late":  [hit_late_mean.get(n.split("/")[-1], np.nan) for n in item_names],
}).dropna()

print(f"kept {len(df)} items out of {len(name_to_idx_)} total ({100*len(df)/len(name_to_idx_):.1f}%)")

In [ ]:
def corr_report(x, y, label):
    r_s, _ = spearmanr(x, y)
    r_p, _ = pearsonr(x, y)
    print(f"{label:<15}  Spearman r = {r_s:+.3f}   Pearson r = {r_p:+.3f}")
    return r_s, r_p

# --- false alarms ---
#r_fa_early = corr_report(df["typicality"], df["fa_early"], "FA")
r_fa  = corr_report(df["typicality"], df["fa"],  "FA")

# --- hits ---
r_hit_early = corr_report(df["typicality"], df["hit_early"], "Hit early (ISI=0-8)")
r_hit_late  = corr_report(df["typicality"], df["hit_late"],  "Hit late (ISI=16-64)")

df["hit_diff"] = df["hit_late"] - df["hit_early"]
r_hit_diff     = corr_report(df["typicality"], df["hit_diff"], "Hit (late−early)")

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(1, 2, figsize=(10,5))

# --- false alarms ---
axs[0].scatter(df["typicality"], df["fa"], alpha=0.4, color="red", label=f"R={r_fa[0]:.3f}")
axs[0].set_xlabel(f"Typicality (−mean kNN distance [K={k}])")
axs[0].set_ylabel("False alarm rate")
axs[0].set_title("False Alarms vs Typicality")
axs[0].legend()
axs[0].grid(alpha=0.3)

# --- hits ---
axs[1].scatter(df["typicality"], df["hit_early"], alpha=0.4, color="gray", label=f"Early (ISI=0-8): R={r_hit_early[0]:.3f}")
axs[1].scatter(df["typicality"], df["hit_late"],  alpha=0.6, color="green", label=f"Late (ISI=16-64): R={r_hit_late[0]:.3f}")
axs[1].set_xlabel(f"Typicality (−mean kNN distance [K={k}])")
axs[1].set_ylabel("Hit rate")
axs[1].set_title("Hits vs Typicality")
axs[1].legend()
axs[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/figures/human-results/itemwise-rates_by_typicality.png")
plt.show()